# LLM-Driven VLM–RAG Integration for Building Management System Control
This notebook demonstrates -

- a modular Python implementation demonstrating an architecture that combines Vision-Language Model (VLM) frame analysis, a RAG context retriever for Safety Data Sheets (SDS) and HVAC specs, and an LLM decision agent that outputs BACnet control commands.

1. **Vision Ingestion (`VLMFrameAnalyzer`):**  
   Analyzes camera frames to extract high-level semantic observations (e.g., identifying a floor-polishing machine and chemical solvents) rather than just raw pixels.

2. **Context Retrieval (`BuildingKnowledgeRAG`):**  
   Queries a vector database using the VLM's extracted labels to pull the exact Safety Data Sheets (SDS) for chemical hazards and the specific HVAC unit capabilities for that zone.

3. **Synthesis & Control Decision (`DynamicIEQAgent`):**  
   Merges the SDS requirements (e.g., "do not recirculate"), HVAC hardware constraints (e.g., available economizers), and real-time sensor levels ($3,450\text{ ppb TVOC}$) to generate an isolated purge command.

4. **Hardware Execution (`BACnetInterface`):**  
   Converts the JSON-like command into physical register overrides (damper percentages, fan speeds, relay triggers).

In [1]:
import json
import time
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional

In [2]:
# =====================================================================
# 1. SCHEMAS & DATA STRUCTURES
# =====================================================================

@dataclass
class VisualObservation:
    """Structured output generated by the VLM from camera feed analysis."""
    detected_objects: List[str]      # e.g., ["cleaning_cart", "floor_buffer"]
    suspected_chemicals: List[str]  # e.g., ["citrus_degreaser", "floor_wax"]
    occupants_present: bool
    confidence_score: float

@dataclass
class SensorTelemetry:
    """Real-time indoor air quality telemetry."""
    zone_id: str
    tvoc_ppb: float
    co2_ppm: float
    pm25_ug_m3: float

@dataclass
class RAGContext:
    """Domain knowledge retrieved from vector database."""
    sds_guidelines: List[str]
    hvac_capabilities: Dict[str, Any]
    ieq_thresholds: Dict[str, float]

@dataclass
class BACnetCommand:
    """BACnet control action output for Building Automation System."""
    ahu_id: str
    outdoor_air_damper_pct: float   # 0.0 to 100.0
    supply_fan_speed_pct: float     # 0.0 to 100.0
    recirculation_damper_pct: float # 0.0 to 100.0
    activated_carbon_scrubber: bool
    system_mode: str                # e.g., "PURGE", "RECIRCULATE", "NORMAL"
    reasoning: str


In [3]:
# =====================================================================
# 2. VISION-LANGUAGE MODEL (VLM) LAYER
# =====================================================================

class VLMFrameAnalyzer:
    """Simulates an LLM Vision pipeline (e.g., GPT-4o Vision or visual-language model)."""
    
    def analyze_zone_image(self, image_frame_bytes: bytes) -> VisualObservation:
        """
        In production, this converts image frame bytes to base64 and passes a prompt to a VLM:
        'Analyze this facility camera feed. Identify janitorial equipment, chemical containers, 
        renovation materials, or hazardous activities. Output structured JSON.'
        """
        # Mock VLM detection output
        return VisualObservation(
            detected_objects=["industrial_cleaning_cart", "floor_polishing_machine", "yellow_mop_bucket"],
            suspected_chemicals=["solvent_floor_stripper", "d_limonene_cleaner"],
            occupants_present=True,
            confidence_score=0.94
        )

In [4]:
# =====================================================================
# 3. VECTOR RAG PIPELINE
# =====================================================================

class BuildingKnowledgeRAG:
    """
    Simulates a Vector DB (e.g., ChromaDB, Pinecone, or FAISS) storing:
    1. Safety Data Sheets (SDS) for facility chemicals
    2. HVAC equipment specs and BACnet mapping tables
    3. ASHRAE / WELL building standard documents
    """
    
    def query_chemical_and_hvac_knowledge(
        self, 
        chemicals: List[str], 
        zone_id: str
    ) -> RAGContext:
        """Queries vector DB based on VLM visual findings and zone context."""
        
        # 1. Semantic search over SDS PDF embeddings
        sds_results = [
            f"SDS for {chemicals[0]}: High vapor pressure solvent. Exposure to indoor concentrations "
            f"exceeding 1000 ppb TVOC requires localized exhaust or outdoor air dilution flush.",
            f"SDS Flash Point / Hazard: Non-flammable at ambient room temps. Do not recirculate vapors."
        ]
        
        # 2. Lookup HVAC equipment specs for the zone
        hvac_specs = {
            "ahu_id": "AHU_Zone_4",
            "max_cfm": 5000,
            "has_outdoor_economizer": True,
            "has_activated_carbon_bed": True,
            "bacnet_points": {
                "damper_point": "AV:401",
                "fan_point": "AV:402"
            }
        }
        
        # 3. Lookup WELL v2 / ASHRAE 62.1 thresholds
        thresholds = {
            "tvoc_max_ppb": 500.0,
            "tvoc_hazard_ppb": 2000.0,
            "co2_max_ppm": 1000.0
        }
        
        return RAGContext(
            sds_guidelines=sds_results,
            hvac_capabilities=hvac_specs,
            ieq_thresholds=thresholds
        )

In [5]:
# =====================================================================
# 4. LLM DECISION AGENT (Reasoning & Orchestration)
# =====================================================================

class DynamicIEQAgent:
    """Synthesizes VLM outputs, real-time telemetry, and RAG context to formulate BACnet commands."""
    
    def evaluate_and_act(
        self, 
        telemetry: SensorTelemetry, 
        visuals: VisualObservation, 
        rag_context: RAGContext
    ) -> BACnetCommand:
        
        # Multi-variable decision logic powered by LLM agent
        tvoc_level = telemetry.tvoc_ppb
        hazard_limit = rag_context.ieq_thresholds["tvoc_hazard_ppb"]
        
        # Condition: High VOC spike combined with visual confirmation of solvent floor stripping
        if tvoc_level > hazard_limit and "floor_polishing_machine" in visuals.detected_objects:
            
            # Reasoning derived from SDS + HVAC specs:
            # - SDS states "Do not recirculate vapors"
            # - HVAC spec shows economizer and carbon bed exist
            action = BACnetCommand(
                ahu_id=rag_context.hvac_capabilities["ahu_id"],
                outdoor_air_damper_pct=100.0,       # Open full fresh air purge
                supply_fan_speed_pct=90.0,          # High exchange rate
                recirculation_damper_pct=0.0,        # Isolate cross-zone return air
                activated_carbon_scrubber=True,     # Scrub volatile organic compounds
                system_mode="OUTDOOR_AIR_PURGE",
                reasoning=(
                    f"VLM confirmed floor stripping activity in {telemetry.zone_id}. "
                    f"TVOC level ({tvoc_level} ppb) exceeds hazard limit ({hazard_limit} ppb). "
                    f"Executed 100% Outdoor Air Purge and zeroed recirculation per SDS instructions."
                )
            )
            return action
            
        # Default baseline control
        return BACnetCommand(
            ahu_id=rag_context.hvac_capabilities["ahu_id"],
            outdoor_air_damper_pct=20.0,
            supply_fan_speed_pct=50.0,
            recirculation_damper_pct=80.0,
            activated_carbon_scrubber=False,
            system_mode="NORMAL",
            reasoning="Normal IEQ operating conditions."
        )


In [6]:
# =====================================================================
# 5. BACNET EXECUTION WRAPPER (Hardware Interface)
# =====================================================================

class BACnetInterface:
    """Translates high-level agent commands into BACnet IP writes."""
    
    def transmit(self, command: BACnetCommand) -> bool:
        print(f"\n[BACnet Driver] Transmitting to AHU Controller: {command.ahu_id}")
        print(f"  ├── System Mode          : {command.system_mode}")
        print(f"  ├── Outdoor Air Damper   : {command.outdoor_air_damper_pct}%")
        print(f"  ├── Recirculation Damper : {command.recirculation_damper_pct}%")
        print(f"  ├── Supply Fan Speed     : {command.supply_fan_speed_pct}%")
        print(f"  └── Carbon Filter Relay  : {'ACTIVE' if command.activated_carbon_scrubber else 'OFF'}")
        print(f"  └── Audit Log            : {command.reasoning}")
        return True

In [7]:
# =====================================================================
# 6. END-TO-END PIPELINE EXECUTION
# =====================================================================

if __name__ == "__main__":
    print("Initializing Multi-Modal VLM + RAG HVAC Orchestrator...")
    
    # 1. Instantiate Pipeline Components
    vlm_analyzer = VLMFrameAnalyzer()
    rag_db = BuildingKnowledgeRAG()
    ieq_agent = DynamicIEQAgent()
    bacnet_driver = BACnetInterface()
    
    # 2. Simulate Input Streams
    simulated_image_frame = b"\x89PNG\r\n\x1a\n..." # Dummy frame bytes
    sensor_stream = SensorTelemetry(
        zone_id="Zone_4_Floor_2",
        tvoc_ppb=3450.0,  # Critical TVOC Spike
        co2_ppm=480.0,    # Low occupancy CO2
        pm25_ug_m3=12.0
    )
    
    # 3. Step 1: Execute VLM Vision Analysis
    print("\nStep 1: Processing Camera Frame via VLM...")
    visual_findings = vlm_analyzer.analyze_zone_image(simulated_image_frame)
    print(f"  Detected Entities : {visual_findings.detected_objects}")
    print(f"  Suspected Sources : {visual_findings.suspected_chemicals}")
    
    # 4. Step 2: Query RAG Pipeline
    print("\nStep 2: Retrieving Domain Context via Vector RAG...")
    rag_context = rag_db.query_chemical_and_hvac_knowledge(
        chemicals=visual_findings.suspected_chemicals,
        zone_id=sensor_stream.zone_id
    )
    print(f"  Retrieved SDS Guidelines: {rag_context.sds_guidelines[0][:80]}...")
    
    # 5. Step 3: LLM Synthesis & Decision
    print("\nStep 3: Evaluating Multi-Modal Strategy in LLM Agent...")
    control_action = ieq_agent.evaluate_and_act(
        telemetry=sensor_stream,
        visuals=visual_findings,
        rag_context=rag_context
    )
    
    # 6. Step 4: Dispatch to BACnet Hardware
    print("\nStep 4: Executing BACnet Hardware Writes...")
    bacnet_driver.transmit(control_action)

Initializing Multi-Modal VLM + RAG HVAC Orchestrator...

Step 1: Processing Camera Frame via VLM...
  Detected Entities : ['industrial_cleaning_cart', 'floor_polishing_machine', 'yellow_mop_bucket']
  Suspected Sources : ['solvent_floor_stripper', 'd_limonene_cleaner']

Step 2: Retrieving Domain Context via Vector RAG...
  Retrieved SDS Guidelines: SDS for solvent_floor_stripper: High vapor pressure solvent. Exposure to indoor ...

Step 3: Evaluating Multi-Modal Strategy in LLM Agent...

Step 4: Executing BACnet Hardware Writes...

[BACnet Driver] Transmitting to AHU Controller: AHU_Zone_4
  ├── System Mode          : OUTDOOR_AIR_PURGE
  ├── Outdoor Air Damper   : 100.0%
  ├── Recirculation Damper : 0.0%
  ├── Supply Fan Speed     : 90.0%
  └── Carbon Filter Relay  : ACTIVE
  └── Audit Log            : VLM confirmed floor stripping activity in Zone_4_Floor_2. TVOC level (3450.0 ppb) exceeds hazard limit (2000.0 ppb). Executed 100% Outdoor Air Purge and zeroed recirculation per SDS inst

## Python Code Architecture with Multi-Zone Resolution & BAC0

In [8]:
import time
import logging
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional

In [9]:
# Import BAC0 for native BACnet/IP communication
try:
    import BAC0
except ImportError:
    BAC0 = None  # Fallback type handling if BAC0 isn't installed in environment

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("HVAC_Orchestrator")

# =====================================================================
# 1. SCHEMAS & MULTI-ZONE DATA MODELS
# =====================================================================

@dataclass
class ZoneTelemetry:
    zone_id: str
    vav_id: str
    tvoc_ppb: float
    co2_ppm: float
    temperature_c: float
    target_temp_c: float = 21.5
    is_occupied: bool = True

@dataclass
class VAVControlAction:
    vav_id: str
    damper_position_pct: float     # 0-100%
    reheat_valve_pct: float        # 0-100% (offsets cold purge air)
    priority: int = 8               # BACnet Priority 8 (Manual/Agent Override)

@dataclass
class AHUControlAction:
    ahu_id: str
    outdoor_air_damper_pct: float  # 0-100%
    supply_fan_speed_pct: float    # 0-100%
    recirc_damper_pct: float       # 0-100%
    vav_actions: Dict[str, VAVControlAction] = field(default_factory=dict)
    system_mode: str = "AUTO"
    reasoning: str = ""

In [10]:
# =====================================================================
# 2. MULTI-ZONE CONFLICT RESOLUTION ENGINE
# =====================================================================

class MultiZoneConflictResolver:
    """
    Resolves competing zone demands served by a single Air Handling Unit (AHU).
    Hierarchy of Priority: Health/Safety Hazard > Thermal Comfort > Energy Optimization.
    """
    
    TVOC_HAZARD_THRESHOLD = 2000.0  # ppb
    TVOC_ELEVATED_THRESHOLD = 800.0  # ppb
    
    def resolve_ahu_zones(self, ahu_id: str, zone_states: List[ZoneTelemetry]) -> AHUControlAction:
        hazard_zones = [z for z in zone_states if z.tvoc_ppb >= self.TVOC_HAZARD_THRESHOLD]
        cold_zones = [z for z in zone_states if z.temperature_c < (z.target_temp_c - 1.5)]
        
        vav_commands: Dict[str, VAVControlAction] = {}
        
        # -------------------------------------------------------------
        # SCENARIO A: ACUTE HAZARD IN AT LEAST ONE ZONE
        # -------------------------------------------------------------
        if hazard_zones:
            hazard_ids = [z.zone_id for z in hazard_zones]
            logger.warning(f"Hazardous TVOC detected in zones: {hazard_ids}. Triggering Purge.")
            
            # AHU Level: Open Outdoor Air fully to dilute chemical hazard
            ahu_action = AHUControlAction(
                ahu_id=ahu_id,
                outdoor_air_damper_pct=100.0,
                supply_fan_speed_pct=90.0,
                recirc_damper_pct=0.0,
                system_mode="HAZARD_PURGE",
                reasoning=f"Acute TVOC spike in {hazard_ids}. AHU forced to 100% Outdoor Air Purge."
            )
            
            # VAV Level Branch Compensation:
            for zone in zone_states:
                if zone.tvoc_ppb >= self.TVOC_HAZARD_THRESHOLD:
                    # Contaminated Zone: Flush maximum air
                    vav_commands[zone.vav_id] = VAVControlAction(
                        vav_id=zone.vav_id,
                        damper_position_pct=100.0,
                        reheat_valve_pct=0.0
                    )
                else:
                    # Clean Neighbor Zone: Throttle damper to prevent cross-contamination,
                    # and engage reheat valve to protect thermal comfort against cold purge air.
                    reheat_needed = 60.0 if zone in cold_zones else 20.0
                    vav_commands[zone.vav_id] = VAVControlAction(
                        vav_id=zone.vav_id,
                        damper_position_pct=25.0,  # Maintain minimum hygienic ventilation
                        reheat_valve_pct=reheat_needed
                    )
            
            ahu_action.vav_actions = vav_commands
            return ahu_action

        # -------------------------------------------------------------
        # SCENARIO B: BALANCED / STANDARD OPERATION
        # -------------------------------------------------------------
        ahu_action = AHUControlAction(
            ahu_id=ahu_id,
            outdoor_air_damper_pct=25.0,
            supply_fan_speed_pct=60.0,
            recirc_damper_pct=75.0,
            system_mode="BALANCED",
            reasoning="All zones within safety limits. Operating in standard economizer mode."
        )
        
        for zone in zone_states:
            vav_commands[zone.vav_id] = VAVControlAction(
                vav_id=zone.vav_id,
                damper_position_pct=50.0,
                reheat_valve_pct=0.0
            )
            
        ahu_action.vav_actions = vav_commands
        return ahu_action

In [11]:
# =====================================================================
# 3. BAC0 HARDWARE INTERFACE (Real BACnet/IP Execution)
# =====================================================================

class BACnetHardwareManager:
    """
    Manages physical BACnet/IP communication using BAC0 stack.
    Handles Point Writing via Priority Array Overrides.
    """
    
    def __init__(self, local_ip: str = "192.168.1.100/24", port: int = 47808):
        self.local_ip = local_ip
        self.port = port
        self.bacnet = None
        
    def start(self):
        """Initializes the BACnet IP stack."""
        if BAC0 is None:
            logger.error("BAC0 library not installed. Running in simulation mode.")
            return

        try:
            logger.info(f"Connecting to BACnet network on {self.local_ip}:{self.port}...")
            self.bacnet = BAC0.connect(ip=self.local_ip, port=self.port)
            logger.info("BACnet/IP Stack initialized successfully.")
        except Exception as e:
            logger.error(f"Failed to start BAC0 stack: {e}")

    def write_point(self, target_ip: str, object_type: str, instance: int, value: float, priority: int = 8):
        """
        Executes BACnet WriteProperty over IP.
        Example syntax for BAC0: bacnet.write('192.168.1.50 analogOutput 1 presentValue 100.0 - 8')
        """
        write_command = f"{target_ip} {object_type} {instance} presentValue {value} - {priority}"
        
        if self.bacnet:
            try:
                self.bacnet.write(write_command)
                logger.info(f"[BACnet Write SUCCESS] -> {write_command}")
            except Exception as e:
                logger.error(f"[BACnet Write FAILED] -> {write_command} | Error: {e}")
        else:
            # Simulation log if BAC0 hardware stack is not running
            logger.info(f"[SIMULATED BACnet Write] -> {write_command}")

    def execute_ahu_commands(self, device_ip: str, action: AHUControlAction, point_map: Dict[str, Any]):
        """Translates high-level AHU and VAV actions into specific BACnet point writes."""
        
        logger.info(f"\n--- Dispatching BACnet Commands to AHU Controller ({device_ip}) ---")
        
        # 1. Write AHU Central Points
        self.write_point(device_ip, "analogOutput", point_map["ahu_oa_damper"], action.outdoor_air_damper_pct)
        self.write_point(device_ip, "analogOutput", point_map["ahu_fan_speed"], action.supply_fan_speed_pct)
        self.write_point(device_ip, "analogOutput", point_map["ahu_recirc_damper"], action.recirc_damper_pct)
        
        # 2. Write Individual VAV Branch Points
        for vav_id, vav_action in action.vav_actions.items():
            vav_ip = point_map["vav_devices"][vav_id]["ip"]
            damper_instance = point_map["vav_devices"][vav_id]["damper_point"]
            reheat_instance = point_map["vav_devices"][vav_id]["reheat_point"]
            
            self.write_point(vav_ip, "analogOutput", damper_instance, vav_action.damper_position_pct, vav_action.priority)
            self.write_point(vav_ip, "analogOutput", reheat_instance, vav_action.reheat_valve_pct, vav_action.priority)

    def release_override(self, target_ip: str, object_type: str, instance: int, priority: int = 8):
        """Releases a BACnet priority level back to schedule control (writes Null/Relinquish)."""
        release_command = f"{target_ip} {object_type} {instance} presentValue null - {priority}"
        if self.bacnet:
            self.bacnet.write(release_command)
            logger.info(f"[BACnet Priority Released] -> {release_command}")

In [12]:
# =====================================================================
# 4. PIPELINE EXECUTION
# =====================================================================

if __name__ == "__main__":
    # 1. Hardware BACnet Mapping Config
    BACNET_POINT_MAP = {
        "ahu_oa_damper": 1,      # Analog Output 1 (Outdoor Air Damper)
        "ahu_fan_speed": 2,      # Analog Output 2 (Supply Fan VFD)
        "ahu_recirc_damper": 3,   # Analog Output 3 (Recirculation Damper)
        "vav_devices": {
            "VAV_401": {"ip": "192.168.1.51", "damper_point": 1, "reheat_point": 2},
            "VAV_402": {"ip": "192.168.1.52", "damper_point": 1, "reheat_point": 2}
        }
    }

    # 2. Simulate Multi-Zone State Streams
    # Scenario: Zone 1 has a severe chemical spike; Zone 2 is cold with normal air quality.
    zone_1 = ZoneTelemetry(
        zone_id="Zone_401_Lab",
        vav_id="VAV_401",
        tvoc_ppb=3800.0,  # CRITICAL SPIKE
        co2_ppm=450.0,
        temperature_c=22.0
    )
    
    zone_2 = ZoneTelemetry(
        zone_id="Zone_402_Office",
        vav_id="VAV_402",
        tvoc_ppb=210.0,    # Normal
        co2_ppm=620.0,
        temperature_c=19.5  # COLD (Below 21.5°C target)
    )

    # 3. Resolve Conflict
    resolver = MultiZoneConflictResolver()
    ahu_control_action = resolver.resolve_ahu_zones("AHU_Central_04", [zone_1, zone_2])

    print(f"\n[Conflict Resolution Strategy]: {ahu_control_action.system_mode}")
    print(f"[Agent Reasoning]: {ahu_control_action.reasoning}\n")

    # 4. Transmit via BAC0 Engine
    bacnet_mgr = BACnetHardwareManager(local_ip="192.168.1.100/24")
    bacnet_mgr.start()
    
    # Execute hardware writes
    bacnet_mgr.execute_ahu_commands(
        device_ip="192.168.1.50", 
        action=ahu_control_action, 
        point_map=BACNET_POINT_MAP
    )

ERROR:HVAC_Orchestrator:BAC0 library not installed. Running in simulation mode.
INFO:HVAC_Orchestrator:
--- Dispatching BACnet Commands to AHU Controller (192.168.1.50) ---
INFO:HVAC_Orchestrator:[SIMULATED BACnet Write] -> 192.168.1.50 analogOutput 1 presentValue 100.0 - 8
INFO:HVAC_Orchestrator:[SIMULATED BACnet Write] -> 192.168.1.50 analogOutput 2 presentValue 90.0 - 8
INFO:HVAC_Orchestrator:[SIMULATED BACnet Write] -> 192.168.1.50 analogOutput 3 presentValue 0.0 - 8
INFO:HVAC_Orchestrator:[SIMULATED BACnet Write] -> 192.168.1.51 analogOutput 1 presentValue 100.0 - 8
INFO:HVAC_Orchestrator:[SIMULATED BACnet Write] -> 192.168.1.51 analogOutput 2 presentValue 0.0 - 8
INFO:HVAC_Orchestrator:[SIMULATED BACnet Write] -> 192.168.1.52 analogOutput 1 presentValue 25.0 - 8
INFO:HVAC_Orchestrator:[SIMULATED BACnet Write] -> 192.168.1.52 analogOutput 2 presentValue 60.0 - 8



[Conflict Resolution Strategy]: HAZARD_PURGE
[Agent Reasoning]: Acute TVOC spike in ['Zone_401_Lab']. AHU forced to 100% Outdoor Air Purge.

